In [1]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import sys
from pathlib import Path
import xarray as xr
import cfgrib
import cartopy.crs as ccrs  # Projeções de mapas.
import cartopy.feature as cfeature  # Elementos geográficos.
from matplotlib.tri import Triangulation

#Import local libraries
import aux

In [2]:
init='2025120100'
year='2025'
month='12'
base_dir='/p/projetos/monan_atm/madeleine.gacita/global_data/'
exps_data={
    "teste_10deep": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/testes_rqccuten/teste_10deep/"},
    "teste_2deep": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/testes_rqccuten/teste_2deep/"},
    "teste_2deep_07liq": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/testes_rqccuten/teste_2deep_07liq/"},
    "teste_2shallow": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/testes_rqccuten/teste_2shallow/"},
    "CTRL": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/testes_rqccuten/CTRL/"},
}
extents={"SA":{
            "extent":[-90, -30, -40, 10],
            "label":"South America"},
        "Global":{
            "extent":[-180, 180, -90, 90],
            "label":"Global"},
        }


In [3]:
var_dict={
    "ISR": {
        "era5_name" : "ssrd",
        "era5_longname" :"surface_solar_radiation_downwards",
        "ceres_name" : "init_all_sfc_sw_dn", 
        "monan_name" : "swdnb",
        "unit" : "W m^{-2}",
        "label" : "Surface shortwave radiation downwards",
        "convert_to_flux": "yes",
        "vmin" : 0,
        "vmax" : 1300,
        "vmin_diff" : -200,
        "vmax_diff" : 200,
    },
    "ISRC": {
        "era5_name" : "ssrdc",
        "era5_longname" :"surface_solar_radiation_downward_clear_sky",
        "ceres_name" : "init_clr_sfc_sw_dn", 
        "monan_name" : "swdnbc",
        "unit" : "W m^{-2}",
        "label" : "Surface shortwave radiation downwards (clear sky)",
        "convert_to_flux": "yes",
        "vmin" : 0,
        "vmax" : 1300,
        "vmin_diff" : -200,
        "vmax_diff" : 200,
    },
    "OLR": {
        "era5_name": "ttr",
        "era5_longname" :"top_net_thermal_radiation",
        "ceres_name" : "init_all_toa_lw_up", 
        "monan_name" : "lwupt",
        "unit" : "W m^{-2}",
        "label" : "TOA Outgoing longwave radiation",
        "convert_to_flux": "yes",
        "vmin" : -400,
        "vmax" : 0
    },
    "OLRC": {
        "era5_name" : "ttrc",
        "era5_longname" : "top_net_thermal_radiation_clear_sky",
        "ceres_name" : "init_clr_toa_lw_up", 
        "monan_name" : "lwuptc",
        "unit" : "W m^{-2}",
        "label" : "TOA Outgoing longwave radiation (clear sky)",
        "convert_to_flux": "yes",
        "vmin" : -400,
        "vmax" : 0,
        "vmin_diff" : -50,
        "vmax_diff" : 50
    },
    "TISR": {
        "era5_name" : "tisr",
        "era5_longname" : "toa_incident_solar_radiation",
        "ceres_name" : "toa_sw_insol", 
        "monan_name" : "swdnt",
        "unit" : "W m^{-2}",
        "label" : "TOA incident short-wave (solar) radiation",
        "convert_to_flux": "yes",
        "vmin" : 0,
        "vmax" : 1300,
        "vmin_diff" : -50,
        "vmax_diff" : 50
    },
    "PC": {
        "era5_name" : "tclw",
        "era5_longname" :"total_column_cloud_liquid_water",
        "ceres_name" : "obs_cld_lwp", 
        "monan_name" : "precipcloud",
        "unit" : "kg*m^{-2}",
        "label" : "Total column cloud liquid water",
        "vmin" : 0.05,
        "vmax" : 0.5,
        "vmin_diff" : -2,
        "vmax_diff" : 2,
        "convert_to_flux": "no",
    },
    "PI": {
        "era5_name": "tciw",
        "era5_longname" :"total_column_cloud_ice_water",
        "ceres_name" : "obs_cld_iwp", 
        "monan_name" : "precipice",
        "unit" : "kg*m^{-2}",
        "label" : "Total column cloud ice water",
        "vmin" : 0.005,
        "vmax" : 0.5,
        "vmin_diff" : -0.5,
        "vmax_diff" : 0.5,
        "convert_to_flux": "no",
    },
    "PI+PC": {
        "era5_name" : ["tciw", "tclw"],
        "era5_longname" : ["total_column_cloud_ice_water", "total_column_cloud_liquid_water"],
        "ceres_name" : ["obs_cld_iwp", "obs_cld_lwp"], 
        "monan_name" : ["precipice", "precipcloud"],
        "unit" : "kg*m^{-2}",
        "label" : "Total column cloud condensate (ice + liquid)",
        "vmin" : 0.05,
        "vmax" : 1.0,
        "vmin_diff" : -1.0,
        "vmax_diff" : 1.0,
        "convert_to_flux": "no",
    },
    "PW": {
        "era5_name" : "tcwv",
        "era5_longname" :"total_column_water_vapour",
        "ceres_name" : "adj_pw", 
        "monan_name" : "precipw",
        "unit" : "kg*m^{-2}",
        "label" : "Precipitable water",
        "vmin" : 10,
        "vmax" : 60,
        "vmin_diff" : -20,
        "vmax_diff" : 20,
        "convert_to_flux": "no",
    },
     "RAINC": {
        "era5_name" : "cp",
        "era5_longname" : "convective_precipitation",
        "ceres_name" : "", 
        "monan_name" : "rainc",
        "unit" : "mm",
        "label" : "Convective precipitation",
        "vmin" : 1,
        "vmax" : 200,
        "vmin_diff" : -100,
        "vmax_diff" : 100,
        "convert_to_flux": "no",
    }, 
     "RAIN": {
        "era5_name" : "tp",
        "era5_longname" : "gridbox_precipitation",
        "ceres_name" : "", 
        "monan_name" : "rainnc",
        "unit" : "mm",
        "label" : "Total precipitation",
        "vmin" : 1,
        "vmax" : 200,
        "vmin_diff" : -100,
        "vmax_diff" : 100,
        "convert_to_flux": "no",
    },  
    "CAPE": {
        "era5_name" : "cape",
        "era5_longname" : "convective_available_potential_energy",
        "ceres_name" : "cape", 
        "monan_name" : "cape",
        "unit" : "J kg^{-1}",
        "label" : "Convective available potential energy",
        "convert_to_flux": "no",
        "vmin" : 0,
        "vmax" : 2000
    },
    "CIN": {
        "era5_name" : "cin",
        "era5_longname" : "convective_inhibition",
        "ceres_name" : "cin", 
        "monan_name" : "cin",
        "unit" : "J kg^{-1}",
        "label" : "Convective inhibition",
        "convert_to_flux": "no",
        "vmin" : 0,
        "vmax" : 300
    }
}

surf_flux_dict={
    "HF": {
        "era5_name" : "sshf",
        "era5_longname" : "surface_sensible_heat_flux",
        "monan_name" : "hfx", 
        "unit" : "W/m**2",
        "label" : "Sensible heat flux"
    },
    "LF": {
        "era5_name": "sslf",
        "era5_longname":"surface_latent_heat_flux",
        "monan_name": "lf", 
        "unit": "W m^{-2}",
        "label": "Latent heat flux"
    }
}

profile_vars_dict={
    "ISR": {
        "monan_name" : "ssrd",
        "ceres_name" : "adj_all_sw_dn",
        "unit" : "W m^{-2}",
        "label" : "Incoming shortwave radiation"
    },
    "OSR": {
        "monan_name" : "ssrd",
        "ceres_name" : "adj_all_sw_up", 
        "unit" : "W m^{-2}",
        "label" : "Outgoing shortwave radiation"
    },
    "ILR": {
        "monan_name" : "ttrc",
        "ceres_name" : "adj_all_lw_dn", 
        "unit" : "W m^{-2}",
        "label" : "Incoming longwave radiation"
    },
    "OLR": {
        "monan_name" : "ttr",
        "ceres_name" : "adj_all_lw_up", 
        "unit" : "W m^{-2}",
        "label" : "Outgoing longwave radiation"
    }    
}

### Plots general settings

In [4]:
### Plots general settings
## Color scale precipitation
# from matplotlib.colors import ListedColormap, BoundaryNorm # Lista de Cores 
# cores_legenda_rgb = [
# (255, 255, 255),  # Branco
# (220, 220, 220),  # Cinza Claro
# (180, 180, 180),  # Cinza
# (20, 0, 150),     # Azul Marinho/Roxo
# (0, 0, 255),      # Azul
# (0, 100, 100),    # Verde Escuro/Azul Petr�leo
# (0, 200, 0),      # Verde
# (150, 255, 0),    # Verde Lim�o/Ciano
# (255, 255, 0),    # Amarelo Claro
# (255, 220, 0),    # Amarelo Escuro/Ouro
# (255, 130, 0),    # Laranja
# (230, 25, 25),    # Vermelho Claro
# (100, 0, 0),      # Vermelho Escuro/Borgonha
# ]

# cores_normalizadas_matplotlib = []
# for r, g, b in cores_legenda_rgb:
#     cores_normalizadas_matplotlib.append((r / 255.0, g / 255.0, b / 255.0))

# # Criacao da colormap e os niveis (clevs)
# cmap = ListedColormap(cores_normalizadas_matplotlib)
# clevs = [0, 1, 2, 4, 6, 10, 15, 25, 35, 50, 75, 100, 150]
# clevs_full = clevs + [40]
# norm = BoundaryNorm(clevs, ncolors=len(clevs), extend='max')

In [5]:
var="PI+PC"
if var_dict[var]["convert_to_flux"]=="yes":
    era_var_name = f'{var_dict[var]["era5_name"]}_flux'
else:
    era_var_name = var_dict[var]["era5_name"]
fig_path="/p/projetos/monan_atm/madeleine.gacita/figuras/"

target_lon = -60.0

# Opening CERES SYN_1deg Ed A

In [6]:
# Print available CERES variables from the first matching file
ceres_dir = Path(base_dir) / "CERES" / "CER_SYN1deg-1Hour"
pattern = f"*{year}{month}*.hdf"
ceres_files = sorted(ceres_dir.glob(pattern))

if not ceres_files:
    print(f"No CERES files found in {ceres_dir} with pattern {pattern}")
else:
    sample_file = ceres_files[0]
    print(f"Sample CERES file: {sample_file.name}")

    ds_tmp = None
    for eng in [None, "netcdf4", "h5netcdf", "scipy"]:
        try:
            ds_tmp = xr.open_dataset(sample_file) if eng is None else xr.open_dataset(sample_file, engine=eng)
            print(f"Opened with xarray engine: {eng or 'default'}")
            break
        except Exception:
            continue

    if ds_tmp is not None:
        print("CERES variables (xarray data_vars):")
        for name in sorted(ds_tmp.data_vars):
            print(f"  - {name}")
        ds_tmp.close()
    else:
        from pyhdf.SD import SD, SDC
        sd = SD(str(sample_file), SDC.READ)
        print("CERES variables (HDF datasets):")
        for name in sorted(sd.datasets().keys()):
            print(f"  - {name}")

Sample CERES file: CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251201.hdf
Opened with xarray engine: netcdf4
CERES variables (xarray data_vars):
  - adj_all_lw_dn
  - adj_all_lw_up
  - adj_all_sfc_spec_lw_dn
  - adj_all_sfc_spec_lw_up
  - adj_all_sfc_spec_sw_dn
  - adj_all_sfc_spec_sw_up
  - adj_all_sw_dn
  - adj_all_sw_up
  - adj_all_toa_spec_lw_up
  - adj_all_toa_spec_sw_dn
  - adj_all_toa_spec_sw_up
  - adj_all_toa_wn
  - adj_allnoaero_lw_dn
  - adj_allnoaero_lw_up
  - adj_allnoaero_sw_dn
  - adj_allnoaero_sw_up
  - adj_cld_amount
  - adj_cld_iwp
  - adj_cld_lwp
  - adj_cld_od
  - adj_cld_temp
  - adj_clr_lw_dn
  - adj_clr_lw_up
  - adj_clr_sw_dn
  - adj_clr_sw_up
  - adj_clr_toa_wn
  - adj_match_aod55
  - adj_pristine_lw_dn
  - adj_pristine_lw_up
  - adj_pristine_sw_dn
  - adj_pristine_sw_up
  - adj_pw
  - adj_sfc_alb
  - adj_skin_temp
  - adj_uth
  - all_sfc_par_diff
  - all_sfc_par_dir
  - all_sfc_sw_diff
  - all_sfc_sw_dir
  - all_sfc_uv_index
  - all_sfc_uva
  - all_

In [7]:
ceres_dir = Path(base_dir) / "CERES" / "CER_SYN1deg-1Hour"
pattern = f"*{year}{month}*.hdf"
ceres_files = sorted(ceres_dir.glob(pattern))

print(f"Searching files in: {ceres_dir}")
print(f"Pattern: {pattern}")

# Use CERES variable from dictionary
ceres_var_name = var_dict[var]["ceres_name"]
if not ceres_var_name:
    raise ValueError(f"var_dict['{var}']['ceres_name'] is empty.")

if isinstance(ceres_var_name, str):
    ceres_var_names = [ceres_var_name]
else:
    ceres_var_names = list(ceres_var_name)

# Explicitly keep only December days 1 to 5
ceres_selected = []
for f in ceres_files:
    date_token = f.name.split(".")[-2]
    file_date = pd.to_datetime(date_token, format="%Y%m%d")
    if file_date.year == int(year) and file_date.month == int(month) and 1 <= file_date.day <= 5:
        ceres_selected.append(f)

ceres_first5 = sorted(ceres_selected)

if len(ceres_first5) != 5:
    raise FileNotFoundError(
        f"Expected 5 CERES files for {year}-{month} days 01..05, found {len(ceres_first5)}."
    )

print("Using files (December days 1-5):")
for f in ceres_first5:
    print(f"  - {f.name}")

# Build xarray DataArray from CERES HDF files (day by day)
daily_means = []
day_labels = []

for ceres_path in ceres_first5:
    print(f"\nOpening: {ceres_path.name}")

    ds_xr = None
    for eng in [None, "netcdf4", "h5netcdf", "scipy"]:
        try:
            if eng is None:
                ds_try = xr.open_dataset(ceres_path)
            else:
                ds_try = xr.open_dataset(ceres_path, engine=eng)
            ds_xr = ds_try
            print(f"  Opened with xarray engine: {eng or 'default'}")
            break
        except Exception:
            continue

    if ds_xr is None:
        from pyhdf.SD import SD, SDC

        sd = SD(str(ceres_path), SDC.READ)
        datasets_info = sd.datasets()
        for name in ceres_var_names:
            if name not in datasets_info:
                raise KeyError(f"{name} not found in {ceres_path.name}")

        ceres_parts = []
        for name in ceres_var_names:
            arr = sd.select(name).get()
            arr = np.asarray(arr, dtype=np.float64)

            # Handle fill value from metadata when available
            fill_candidates = []
            attrs = sd.select(name).attributes()
            for key in ["_FillValue", "fillvalue", "missing_value"]:
                if key in attrs:
                    fill_candidates.append(attrs[key])
            for fv in fill_candidates:
                arr = np.where(arr == fv, np.nan, arr)

            if arr.ndim != 3:
                raise ValueError(f"Expected 3D CERES array (time, lat, lon), got shape {arr.shape}")
            ceres_parts.append(arr)

        arr_total = np.nansum(np.stack(ceres_parts, axis=0), axis=0)

        ntime, nlat, nlon = arr_total.shape
        lat = np.linspace(-89.5, 89.5, nlat)
        lon = np.linspace(-179.5, 179.5, nlon)

        ds_xr = xr.Dataset(
            {
                "ceres_total_cond": (("time", "latitude", "longitude"), arr_total)
            },
            coords={
                "time": pd.date_range(start=pd.to_datetime(ceres_path.name.split(".")[-2], format="%Y%m%d"), periods=ntime, freq="1h"),
                "latitude": lat,
                "longitude": lon,
            },
        )
        print("  Built xarray Dataset from HDF using pyhdf fallback")

    # Build selected variable (single field or PI+PC sum)
    if len(ceres_var_names) == 1:
        if ceres_var_names[0] not in ds_xr.data_vars:
            raise KeyError(f"{ceres_var_names[0]} not found in xarray dataset for {ceres_path.name}")
        da = ds_xr[ceres_var_names[0]]
    else:
        missing_names = [name for name in ceres_var_names if name not in ds_xr.data_vars]
        if missing_names:
            raise KeyError(f"Missing CERES variables in {ceres_path.name}: {missing_names}")
        da = sum(ds_xr[name] for name in ceres_var_names)

    # Ensure standard dimension names
    rename_map = {}
    if "lat" in da.dims:
        rename_map["lat"] = "latitude"
    if "lon" in da.dims:
        rename_map["lon"] = "longitude"
    if rename_map:
        da = da.rename(rename_map)

    # 24-hour mean: average over all non-spatial dimensions (e.g., time/gmt_hr_index)
    mean_dims = [dim for dim in da.dims if dim not in ["latitude", "longitude"]]
    if not mean_dims:
        raise ValueError(f"No temporal dimension found for CERES variable(s) {ceres_var_names} in {ceres_path.name}")

    day_mean = da.mean(dim=mean_dims, skipna=True)

    daily_means.append(day_mean)
    day_labels.append(pd.to_datetime(ceres_path.name.split(".")[-2], format="%Y%m%d"))

ceres_daily_mean = xr.concat(daily_means, dim="day")
ceres_daily_mean = ceres_daily_mean.assign_coords(day=("day", day_labels))

print(f"\nComputed 24-hour mean for CERES variable(s) {ceres_var_names} (days 1 to 5):")
print(ceres_daily_mean)

Searching files in: /p/projetos/monan_atm/madeleine.gacita/global_data/CERES/CER_SYN1deg-1Hour
Pattern: *202512*.hdf
Using files (December days 1-5):
  - CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251201.hdf
  - CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251202.hdf
  - CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251203.hdf
  - CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251204.hdf
  - CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251205.hdf

Opening: CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251201.hdf
  Opened with xarray engine: netcdf4

Opening: CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251202.hdf
  Opened with xarray engine: netcdf4

Opening: CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251203.hdf
  Opened with xarray engine: netcdf4

Opening: CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Edition4B_415412.20251204.hdf
  Opened with xarray engine: netcdf4

Opening: CER_SYN1deg-1Hour_Terra-Aqua-NOAA20_Ed

In [8]:
# Plot CERES 24-hour means and save figures (same save logic as other cells)

if "ceres_daily_mean" not in globals():
    raise NameError("Run the previous CERES processing cell first.")

if "fig_path" not in globals():
    fig_path = "/p/projetos/monan_atm/madeleine.gacita/figuras/"

# Convert CERES cloud water/ice products from g m^-2 to kg m^-2 for plotting consistency
ceres_plot_scale = 1000.0 if var in ["PI", "PC", "PI+PC"] else 1.0
if ceres_plot_scale != 1.0:
    print("Applying CERES scale factor: dividing plotted values by 1000")

# Use configured limits when available; fallback to robust percentiles
scaled_values = ceres_daily_mean.values / ceres_plot_scale
plot_vmin = float(var_dict[var].get("vmin", np.nanpercentile(scaled_values, 2)))
plot_vmax = float(var_dict[var].get("vmax", np.nanpercentile(scaled_values, 98)))
print(f"Using plot range: vmin={plot_vmin:.6g}, vmax={plot_vmax:.6g}")

# Colormap: values masked (< vmin) appear white
plot_cmap_obj = plt.get_cmap("turbo").copy()
plot_cmap_obj.set_bad("white")
plot_cmap_obj.set_under("white")

for day in pd.to_datetime(ceres_daily_mean["day"].values):
    day_str = day.strftime("%Y-%m-%d")
    day_compact = day.strftime("%Y%m%d")
    print(f"Plotting CERES daily mean for {day_str}")

    data_to_plot = ceres_daily_mean.sel(day=day) / ceres_plot_scale

    # Diagnostic: check values north of 60N before masking
    lat_name = "latitude" if "latitude" in data_to_plot.coords else ("lat" if "lat" in data_to_plot.coords else None)
    if lat_name is not None:
        north60 = data_to_plot.where(data_to_plot[lat_name] >= 60, drop=True)
        north_valid = int(north60.count().item())
        if north_valid > 0:
            north_min = float(north60.min(skipna=True).item())
            north_max = float(north60.max(skipna=True).item())
            north_above_vmin = int((north60 >= plot_vmin).sum(skipna=True).item())
            north_frac_above = 100.0 * north_above_vmin / north_valid
            print(
                f"  North of 60N (pre-mask): min={north_min:.6g}, max={north_max:.6g}, "
                f"valid_points={north_valid}"
            )
            print(
                f"  North of 60N (>=vmin): points={north_above_vmin}/{north_valid} "
                f"({north_frac_above:.2f}%)"
            )
        else:
            print("  North of 60N (pre-mask): no valid points")

    data_to_plot = data_to_plot.where(data_to_plot >= plot_vmin)

    for myext in extents.keys():
        extent_label = extents[myext]["label"]
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

        data_to_plot.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=plot_cmap_obj,
            vmin=plot_vmin,
            vmax=plot_vmax,
            cbar_kwargs={
                "shrink": 0.5,
                "aspect": 25,
                "pad": 0.05,
                "label": f"{var_dict[var]['label']} ({var_dict[var]['unit']})",
                "extend": "max",
            },
        )

        ax.set_extent(extents[myext]["extent"], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3)
        ax.add_feature(cfeature.STATES, linewidth=0.2)

        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": 10}
        gl.ylabel_style = {"size": 10}

        ax.set_title(
            f"CERES {var_dict[var]['label']} daily mean\n{day_str}, {extent_label}",
            fontsize=12,
            pad=20,
        )

        plt.tight_layout()
        out_png = f"{fig_path}/CERES_{var}_daily_mean_{day_compact}_{myext}.png"
        plt.savefig(out_png, dpi=150, bbox_inches="tight")
        print(f"Saved plot to: {out_png}")
        plt.close()

Applying CERES scale factor: dividing plotted values by 1000
Using plot range: vmin=0.05, vmax=1
Plotting CERES daily mean for 2025-12-01
  North of 60N (pre-mask): min=0.0290647, max=2.79495, valid_points=10440
  North of 60N (>=vmin): points=10240/10440 (98.08%)
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//CERES_PI+PC_daily_mean_20251201_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//CERES_PI+PC_daily_mean_20251201_Global.png
Plotting CERES daily mean for 2025-12-02
  North of 60N (pre-mask): min=0.0130783, max=0.836398, valid_points=10602
  North of 60N (>=vmin): points=10374/10602 (97.85%)
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//CERES_PI+PC_daily_mean_20251202_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//CERES_PI+PC_daily_mean_20251202_Global.png
Plotting CERES daily mean for 2025-12-03
  North of 60N (pre-mask): min=0.0181611, max=0.627335, valid_points=10684
  North of 60N (>=vmin): points=10

## Opening ERA5 {var} data

In [9]:
def apply_lon_lat_conventions(ds):
    # Renames
    if "lon" in ds.dims:
        ds = ds.rename({"lon": "longitude"})
    if "lat" in ds.dims:
        ds = ds.rename({"lat": "latitude"})
 
    # Flip latitudes (ensure they are monotonic increasing)
    if "latitude" in ds.dims:
        lats = ds["latitude"]
        if len(lats) > 1 and lats[0] > lats[-1]:
            ds = ds.reindex(latitude=ds.latitude[::-1])
 
    # Convert longitude to [-180, 180[
    if "longitude" in ds.dims and ds["longitude"].max() > 180:
        lons = ds["longitude"]
        lons_attrs = lons.attrs
        new_lons = np.concatenate([lons[lons >= 180], lons[lons < 180]])
        ds = ds.reindex(longitude=new_lons)
        ds = ds.assign_coords(longitude=(((ds["longitude"] + 180) % 360) - 180))
        ds["longitude"].attrs = lons_attrs
    return ds

In [10]:
if var == "PI+PC":
    era5_longnames = var_dict[var]["era5_longname"]
    era5_names = var_dict[var]["era5_name"]
    if len(era5_longnames) != 2 or len(era5_names) != 2:
        raise ValueError("For PI+PC, expected two ERA5 names and longnames")

    era5_path_pi = Path(base_dir + f"era5/single_levels/{year}/{month}/{era5_longnames[0]}.nc")
    era5_path_pc = Path(base_dir + f"era5/single_levels/{year}/{month}/{era5_longnames[1]}.nc")

    if (not os.path.exists(era5_path_pi)) or (not os.path.exists(era5_path_pc)):
        print(f"Missing ERA5 files for PI+PC:\n  PI: {era5_path_pi}\n  PC: {era5_path_pc}")
    else:
        ds_era5_pi = xr.open_dataset(era5_path_pi, engine="netcdf4")
        ds_era5_pc = xr.open_dataset(era5_path_pc, engine="netcdf4")
        ds_era5_pi = apply_lon_lat_conventions(ds_era5_pi)
        ds_era5_pc = apply_lon_lat_conventions(ds_era5_pc)
        ds_era5 = ds_era5_pi
        ds_era5["pi_plus_pc"] = ds_era5_pi[era5_names[0]] + ds_era5_pc[era5_names[1]]
        ds_era5["pi_plus_pc"].attrs = {
            "units": var_dict[var]["unit"],
            "description": "Total column cloud condensate (ERA5 PI + PC)",
        }
        era_var_name = "pi_plus_pc"
        print(ds_era5["pi_plus_pc"].attrs)
else:
    era5_file_arg = base_dir+f"era5/single_levels/{year}/{month}/{var_dict[var]['era5_longname']}.nc"
    era5_path = Path(era5_file_arg)
    if not os.path.exists(era5_path):    
        print(f"File does not exist: {era5_path}")
    else:
        ds_era5 = xr.open_dataset(era5_path, engine="netcdf4")
        ds_era5 = apply_lon_lat_conventions(ds_era5)

        if var_dict[var]["convert_to_flux"]=="yes":
            ds_era5[era_var_name] = ds_era5[var_dict[var]["era5_name"]]/3600
            ds_era5[era_var_name].attrs = {'units': var_dict[var]["unit"]}
            ds_era5[era_var_name].attrs['description'] = f'Instantaneous flux calculated from accumulated {var_dict[var]["era5_name"].upper()}'
            print(ds_era5[era_var_name].attrs)
            print(ds_era5['valid_time'])
        else:
            print(ds_era5[var_dict[var]["era5_name"]].attrs)


{'units': 'kg*m^{-2}', 'description': 'Total column cloud condensate (ERA5 PI + PC)'}


In [11]:
start_date = pd.Timestamp("2025-12-01 00:00:00")
end_date = pd.Timestamp("2025-12-05 23:00:00")

time_name = "valid_time" if "valid_time" in ds_era5.coords else "time"
if var == "PI+PC":
    era_source_name = "pi_plus_pc"
else:
    era_source_name = era_var_name if era_var_name in ds_era5.data_vars else var_dict[var]["era5_name"]
era_field = ds_era5[era_source_name]

era_window = era_field.sel({time_name: slice(start_date, end_date)})

if era_window.sizes.get(time_name, 0) == 0:
    print(f"No ERA5 data found between {start_date} and {end_date}")
else:
    # Same day-based logic used in the rain notebook, but averaging instead of summing
    era_window = era_window.assign_coords(day=era_window[time_name].dt.floor("D"))
    era5_daily_mean = era_window.groupby("day").mean(dim=time_name, skipna=True, keep_attrs=True)
    era5_daily_mean.name = f"{era_source_name}_daily_mean"
    era5_daily_mean.attrs["description"] = (
        f"ERA5 daily mean values for {era_source_name} from {start_date:%Y-%m-%d} to {end_date:%Y-%m-%d}"
    )

    print(era5_daily_mean)

    spatial_dims = [dim for dim in era5_daily_mean.dims if dim != "day"]
    if spatial_dims:
        daily_domain_mean = era5_daily_mean.mean(dim=spatial_dims, skipna=True)
        print("\nDomain-mean value by day:")
        print(daily_domain_mean.to_series())

<xarray.DataArray 'pi_plus_pc_daily_mean' (day: 5, latitude: 721,
                                           longitude: 1440)> Size: 21MB
array([[[6.9936119e-05, 6.9936119e-05, 6.9936119e-05, ...,
         6.9936119e-05, 6.9936119e-05, 6.9936119e-05],
        [2.9500326e-04, 2.9500326e-04, 2.9754639e-04, ...,
         2.9118857e-04, 2.9118857e-04, 2.9246011e-04],
        [3.2043457e-04, 3.2170615e-04, 3.1916299e-04, ...,
         3.1280518e-04, 3.1661987e-04, 3.1534830e-04],
        ...,
        [5.9444427e-02, 5.9426624e-02, 5.9424084e-02, ...,
         5.9477489e-02, 5.9467316e-02, 5.9448242e-02],
        [6.1929066e-02, 6.1931610e-02, 6.1922710e-02, ...,
         6.1954498e-02, 6.1944325e-02, 6.1934154e-02],
        [6.2896729e-02, 6.2896729e-02, 6.2896729e-02, ...,
         6.2896729e-02, 6.2896729e-02, 6.2896729e-02]],

       [[3.0212402e-03, 3.0212402e-03, 3.0212402e-03, ...,
         3.0212402e-03, 3.0212402e-03, 3.0212402e-03],
        [2.8889973e-03, 2.8940837e-03, 2.8940837e

In [ ]:
# Plot ERA5 daily mean for the selected variable, following the same loop logic as compare_rain

if "era5_daily_mean" not in globals():
    raise NameError("Run the previous ERA5 daily-mean cell first.")

# Use CERES color scale (turbo) for consistency across datasets
plot_cmap = "turbo"

# Keep variable-dependent limits; force PI to MONAN limits for comparison
if var == "ISR":
    vmin, vmax = 0, 500
elif var == "OLR":
    vmin, vmax = 0, 250
elif var == "PI":
    vmin, vmax = 0.005, 0.5
elif var == "PI+PC":
    vmin, vmax = 0.005, 1.0
elif var in ["RAIN", "RAINC"]:
    vmin = var_dict[var].get("vmin", float(era5_daily_mean.min().item()))
    vmax = var_dict[var].get("vmax", float(era5_daily_mean.max().item()))
else:
    vmin = var_dict[var].get("vmin", float(era5_daily_mean.min().item()))
    vmax = var_dict[var].get("vmax", float(era5_daily_mean.max().item()))

# Colormap: values masked (< vmin) appear white
plot_cmap_obj = plt.get_cmap(plot_cmap).copy()
plot_cmap_obj.set_bad("white")
plot_cmap_obj.set_under("white")

# Explicit norm ensures the colorbar strictly spans [vmin, vmax]
era5_norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

for day in pd.to_datetime(era5_daily_mean["day"].values):
    day_str = day.strftime("%Y-%m-%d")
    print(f"Plotting ERA5 daily mean for {day_str}")

    data_to_plot = era5_daily_mean.sel(day=day)
    data_to_plot = data_to_plot.where(data_to_plot >= vmin)

    for myext in extents.keys():
        extent_label = extents[myext]["label"]
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

        data_to_plot.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=plot_cmap_obj,
            norm=era5_norm,
            cbar_kwargs={
                "shrink": 0.5,
                "aspect": 25,
                "pad": 0.05,
                "label": f"{var_dict[var]['label']} ({var_dict[var]['unit']})",
                "extend": "max",
            },
        )

        ax.set_extent(extents[myext]["extent"], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3)
        ax.add_feature(cfeature.STATES, linewidth=0.2)

        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": 10}
        gl.ylabel_style = {"size": 10}

        ax.set_title(
            f"ERA5 {var_dict[var]['label']} daily mean\n{day_str}, {extent_label}",
            fontsize=12,
            pad=20,
        )

        plt.tight_layout()
        out_png = f"{fig_path}/ERA5_{var}_daily_mean_{day.strftime('%Y%m%d')}_{myext}.png"
        plt.savefig(out_png, dpi=150, bbox_inches="tight")
        print(f"Saved plot to: {out_png}")
        plt.close()


Plotting ERA5 daily mean for 2025-12-01


/home2/madeleine.gacita/.conda/envs/working_env/lib/python3.14/site-packages/cartopy/mpl/geoaxes.py:291: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return func(self, *args, **kwargs)


Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//ERA5_PI+PC_daily_mean_20251201_SA.png


/home2/madeleine.gacita/.conda/envs/working_env/lib/python3.14/site-packages/cartopy/mpl/geoaxes.py:291: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return func(self, *args, **kwargs)


Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//ERA5_PI+PC_daily_mean_20251201_Global.png
Plotting ERA5 daily mean for 2025-12-02


...

Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//ERA5_PI+PC_daily_mean_20251204_Global.png
Plotting ERA5 daily mean for 2025-12-05


/home2/madeleine.gacita/.conda/envs/working_env/lib/python3.14/site-packages/cartopy/mpl/geoaxes.py:291: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return func(self, *args, **kwargs)


Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//ERA5_PI+PC_daily_mean_20251205_SA.png


/home2/madeleine.gacita/.conda/envs/working_env/lib/python3.14/site-packages/cartopy/mpl/geoaxes.py:291: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return func(self, *args, **kwargs)


Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//ERA5_PI+PC_daily_mean_20251205_Global.png


## Opening MONAN data and extracting {var}

In [13]:
for exp in exps_data.keys():
    exp_name = exp
    print(f"Processing experiment: {exp_name}")
    monan_dir = exps_data[exp]["dir"]

    if var == "PI+PC":
        monan_name_pi, monan_name_pc = var_dict[var]["monan_name"]
        monan_file_pi = f'{monan_dir}/{init}_hourly_{monan_name_pi}.nc'
        monan_file_pc = f'{monan_dir}/{init}_hourly_{monan_name_pc}.nc'
        monan_path_pi = Path(monan_file_pi)
        monan_path_pc = Path(monan_file_pc)

        if os.path.exists(monan_path_pi) and os.path.exists(monan_path_pc):
            ds_pi = xr.open_dataset(monan_path_pi, engine="netcdf4")
            ds_pc = xr.open_dataset(monan_path_pc, engine="netcdf4")
            ds_pi = ds_pi.assign_coords(day=ds_pi['Time'].dt.floor('D'))
            ds_pc = ds_pc.assign_coords(day=ds_pc['Time'].dt.floor('D'))

            var_data = ds_pi[monan_name_pi] + ds_pc[monan_name_pc]
            ds_daily_mean = var_data.groupby('day').mean(dim='Time')
            exps_data[exp]["ds_daily_mean_pi_plus_pc"] = ds_daily_mean

            print(f"  ✓ Calculated 24 h mean for {var}: {monan_name_pi} + {monan_name_pc}")
            print(ds_daily_mean)
        else:
            print(f"  ✗ Missing PI+PC files:\n    {monan_path_pi}\n    {monan_path_pc}")
    else:
        monan_file = f'{monan_dir}/{init}_hourly_{var_dict[var]["monan_name"]}.nc'
        monan_path = Path(monan_file)

        if os.path.exists(monan_path):
            ds = xr.open_dataset(monan_path, engine="netcdf4")
            ds = ds.assign_coords(day=ds['Time'].dt.floor('D'))

            var_data = ds[var_dict[var]['monan_name']]
            
            # Check if variable has 3 dimensions (Time + 2 spatial) - sum vertically first
            if var_data.ndim == 3:
                # Identify the vertical dimension (not Time, lon, lat)
                spatial_dims = [dim for dim in var_data.dims if dim not in ['Time', 'lon', 'lat']]
                if spatial_dims:
                    vert_dim = spatial_dims[0]  # Get the vertical dimension
                    print(f"  ! Variable has 3 dims, summing along vertical dimension: {vert_dim}")
                    var_data = var_data.sum(dim=vert_dim)
            
            ds_daily_mean = var_data.groupby('day').mean(dim='Time')
            exps_data[exp][f"ds_daily_mean_{var_dict[var]['monan_name']}"] = ds_daily_mean

            print(f"  ✓ Calculated 24 h mean for {var}: {var_dict[var]['monan_name']}")
            print(ds_daily_mean)
        else:
            print(f"  ✗ File does not exist: {monan_path!s}")

print('\nProcessing complete!')

Processing experiment: teste_10deep
  ✓ Calculated 24 h mean for PI+PC: precipice + precipcloud
<xarray.DataArray (day: 5, nCells: 655362)> Size: 13MB
array([[7.9915754e-02, 2.8414875e-02, 1.4892818e-03, ..., 4.2808315e-01,
        3.8602695e-01, 3.5139629e-01],
       [4.7040194e-02, 8.2749436e-03, 1.1004961e-01, ..., 3.9811075e-02,
        5.7556432e-02, 6.1223160e-02],
       [5.1377904e-02, 5.9474021e-02, 8.0209589e-03, ..., 4.1230489e-02,
        2.5462382e-02, 2.4680689e-02],
       [5.6211781e-02, 5.3944487e-02, 1.1975061e-03, ..., 5.2444975e-05,
        6.8394846e-05, 6.0074788e-05],
       [3.6664188e-02, 3.8222991e-02, 1.1384095e-03, ..., 1.2467890e-01,
        1.4610581e-01, 1.8540736e-01]], shape=(5, 655362), dtype=float32)
Coordinates:
  * day      (day) datetime64[ns] 40B 2025-12-01 2025-12-02 ... 2025-12-05
    lat      (nCells) float64 5MB 26.57 26.57 90.0 ... -52.78 -52.54 -52.54
    lon      (nCells) float64 5MB -175.0 -103.0 115.0 ... -175.0 -175.2 -174.7
Dimensions 

### Plotting 24h means

In [ ]:
# Plot MONAN 24 h means for the selected variable, following compare_rain_period logic

if var == "PI+PC":
    ds_key = "ds_daily_mean_pi_plus_pc"
else:
    ds_key = f"ds_daily_mean_{var_dict[var]['monan_name']}"
plot_var_key = var

# Use CERES color scale (turbo) for consistency across datasets
plot_cmap = "turbo"

if plot_var_key == "ISR":
    vmin, vmax = 0, 500
elif plot_var_key == "OLR":
    vmin, vmax = 0, 250
elif plot_var_key == "PI":
    # MONAN fixed limits for PI (also used for CERES/ERA5 comparison)
    vmin, vmax = 0.005, 0.5
elif plot_var_key == "PI+PC":
    vmin, vmax = 0.005, 1.0
elif plot_var_key in ["RAIN", "RAINC"]:
    vmin = var_dict[plot_var_key].get('vmin', 0)
    vmax = var_dict[plot_var_key].get('vmax', 200)
else:
    vmin = var_dict[plot_var_key].get('vmin', None)
    vmax = var_dict[plot_var_key].get('vmax', None)

# Explicit norm ensures colorbar strictly spans [vmin, vmax]
norm_plot = mcolors.Normalize(vmin=vmin, vmax=vmax) if (vmin is not None and vmax is not None) else None

# Make masked values (including below-vmin) appear white
plot_cmap_obj = plt.get_cmap(plot_cmap).copy()
plot_cmap_obj.set_bad("white")
plot_cmap_obj.set_under("white")

for exp in exps_data.keys():
    exp_name = exp
    print(f"Plotting for experiment: {exp_name}")

    if ds_key not in exps_data[exp]:
        print(f"  ✗ Skipping {exp_name}: missing {ds_key}")
        continue

    monan_lons = exps_data[exp][ds_key]['lon']
    monan_lats = exps_data[exp][ds_key]['lat']
    tri = Triangulation(monan_lons, monan_lats)

    n_days = exps_data[exp][ds_key].sizes.get('day', exps_data[exp][ds_key].shape[0])

    for myext in extents.keys():
        extent_label = extents[myext]["label"]

        for lead in range(min(5, n_days)):
            fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

            date = pd.to_datetime(init, format='%Y%m%d%H') + pd.Timedelta(hours=lead * 24)
            data_to_plot = exps_data[exp][ds_key].isel(day=lead).values

            data_masked = np.ma.masked_invalid(data_to_plot)
            if vmin is not None:
                data_masked = np.ma.masked_where(data_masked < vmin, data_masked)

            tpc = ax.tripcolor(
                tri,
                data_masked,
                cmap=plot_cmap_obj,
                norm=norm_plot,
                shading="flat"
            )
            ax.set_extent(extents[myext]["extent"], crs=ccrs.PlateCarree())
            # extend='max' only: values below vmin are masked white, no bottom arrow needed
            plt.colorbar(
                tpc,
                ax=ax,
                label=f"{var_dict[plot_var_key]['label']} ({var_dict[plot_var_key]['unit']})",
                shrink=0.5,
                aspect=25,
                pad=0.05,
                extend='max'
            )

            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.add_feature(cfeature.STATES, linewidth=0.2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 10}
            gl.ylabel_style = {'size': 10}

            ax.set_title(
                f"MONAN {exp_name} {date.strftime('%Y-%m-%d')} + {lead * 24} h lead time, {extent_label}\n"
                f"24 h mean {var_dict[plot_var_key]['label']}",
                fontsize=12,
                pad=20
            )

            plt.tight_layout()
            out_png = f"{fig_path}/MONAN_{plot_var_key}_24h_mean_{init}+{lead * 24}h_{exp_name}_{myext}.png"
            plt.savefig(out_png, dpi=150, bbox_inches='tight')
            print(f"Saved plot to: {out_png}")
            plt.close()


Plotting for experiment: teste_10deep
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+0h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+24h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+48h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+72h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+96h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+0h_teste_10deep_Global.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+24h_teste_10deep_Global.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_PI+PC_24h_mean_2025120100+48h_teste_10deep_Global.png


In [15]:
# Close opened xarray datasets

closed = []
failed = []
seen = set()

def try_close(obj, label):
    obj_id = id(obj)
    if obj_id in seen:
        return
    seen.add(obj_id)
    if hasattr(obj, "close") and callable(getattr(obj, "close")):
        try:
            obj.close()
            closed.append(label)
        except Exception as exc:
            failed.append((label, str(exc)))

# Close direct globals
for name, obj in list(globals().items()):
    if isinstance(obj, (xr.Dataset, xr.DataArray)):
        try_close(obj, name)

# Close xarray objects stored inside dictionaries (e.g., exps_data)
for name, obj in list(globals().items()):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (xr.Dataset, xr.DataArray)):
                try_close(value, f"{name}[{key!r}]")
            elif isinstance(value, dict):
                for subkey, subvalue in value.items():
                    if isinstance(subvalue, (xr.Dataset, xr.DataArray)):
                        try_close(subvalue, f"{name}[{key!r}][{subkey!r}]")

print(f"Closed objects: {len(closed)}")
for item in closed:
    print(f"  - {item}")

if failed:
    print(f"\nFailed to close: {len(failed)}")
    for label, err in failed:
        print(f"  - {label}: {err}")

Closed objects: 22
  - ds_tmp
  - ds_xr
  - da
  - day_mean
  - ceres_daily_mean
  - north60
  - ds_era5_pi
  - ds_era5_pc
  - era_field
  - era_window
  - era5_daily_mean
  - daily_domain_mean
  - ds_pi
  - ds_pc
  - var_data
  - ds_daily_mean
  - monan_lons
  - monan_lats
  - exps_data['teste_10deep']['ds_daily_mean_pi_plus_pc']
  - exps_data['teste_2deep']['ds_daily_mean_pi_plus_pc']
  - exps_data['teste_2deep_07liq']['ds_daily_mean_pi_plus_pc']
  - exps_data['teste_2shallow']['ds_daily_mean_pi_plus_pc']
